In [ ]:
import pandas as pd
import re
import numpy as np
import math

from nltk.corpus import stopwords
# nltk.download('stopwords')
import contractions
import fasttext

from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from umap import UMAP
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

C:\Users\kylie\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Preprocessing step

In [2]:
# Load csv file into songs_df dataframe
songs_df = pd.read_csv("../../TMdata/songs_with_vader.csv")

print("Songs shape:", songs_df.shape)
print("\nSongs columns:", songs_df.columns.tolist())
songs_df.head()

Songs shape: (520589, 23)

Songs columns: ['id', 'name', 'artists', 'danceability', 'energy', 'loudness', 'speechiness', 'valence', 'tempo', 'lyrics', 'year', 'genre', 'popularity', 'total_artist_followers', 'avg_artist_popularity', 'niche_genres', 'lyrics_clean', 'lyrics_normalized', 'language', 'non_english_ratio', 'word_count', 'vader_compound', 'vader_sentiment']


,id,name,artists,danceability,energy,loudness,speechiness,valence,tempo,lyrics,...,total_artist_followers,avg_artist_popularity,niche_genres,lyrics_clean,lyrics_normalized,language,non_english_ratio,word_count,vader_compound,vader_sentiment
0,0Prct5TDjAnEgIqbxcldY9,!,"[""HELLYEAH""]",0.415,0.605,-11.157,0.0575,0.193,100.059,"He said he came from Jamaica,\nhe owned a coup...",...,769490,52.0,"[""groove metal"", ""metal""]","He said he came from Jamaica,\nhe owned a coup...","he said he came from jamaica, he owned a coupl...",en,0.229474,475,0.9574,Positive
1,2ASl4wirkeYm3OWZxXKYuq,!!,"[""Yxngxr1""]",0.788,0.648,-9.135,0.3150,0.287,79.998,"Fuck the bitch, now she running with my kids\n...",...,143628,45.0,[],"Fuck the bitch, now she running with my kids\n...","fuck the bitch, now she running with my kids a...",en,0.175781,256,0.9982,Positive
2,5tA3ImW310llKo8EMBj2Ga,!!Noble Stabbings!!,"[""Dillinger Four""]",0.171,0.957,-5.749,0.1490,0.349,175.317,You like to stand on the other side\nPoint and...,...,36619,35.0,"[""melodic hardcore"", ""pop punk"", ""punk"", ""skat...",You like to stand on the other side\nPoint and...,you like to stand on the other side point and ...,en,0.127962,211,-0.9868,Negative
3,1xBFhv5faebv3mmwxx7DnS,!Lost!,"[""Ril\u00e8s""]",0.729,0.552,-8.562,0.0650,0.380,86.103,I would like to give you all my time\nI would ...,...,929303,63.0,"[""french rap""]",I would like to give you all my time\nI would ...,i would like to give you all my time i would l...,en,0.111732,358,-0.2434,Negative
4,0gNNToCW3qjabgTyBSjt3H,!Que Vida! - Mono Version,"[""Love""]",0.600,0.540,-11.803,0.0328,0.547,125.898,With pictures and words\nIs this communicating...,...,273598,46.0,"[""acid rock"", ""baroque pop"", ""proto-punk"", ""ps...",With pictures and words\nIs this communicating...,with pictures and words is this communicating?...,en,0.170732,123,-0.8528,Negative


In [3]:
# Inspect structure of the lyrics column
songs_df['lyrics_normalized'][0]

'he said he came from jamaica, he owned a couple acres a couple fake visas \'cause he never got his papers gave up on love fucking with them heart breakers but he was getting money with the movers and the shakers he was mixed with a couple things, bald like a couple rings bricks in the condo and grams to sing-sing left arm, baby mother tatted 5-year bid up north when they ratted anyway i felt him, helped him, put him on lock, seat-belt him took him out to belgium, welcome bitches this pretty, that\'s seldom this box better than the box he was held in i\'m momma in that order, i call him daddy like daughters he like it when i get drunk, but i like it when he be sober that\'s top of the toppa, i never fuck with beginners i let him play with my pussy then lick it off of his fingers, i\'m in the zone they holler at me, but it\'s you, you, this is not high school me and my crew, we can slide through give it to you whenever you want, whip it whenever you want baby it\'s yours anywhere, every

In [4]:
# Preprocessing function
def clean_lyrics(text):
    if not isinstance(text, str):
        return ""
    
    # Replace literal '\n' with actual newlines
    text = text.replace('\\n', '\n')
    
    # Split into lines
    lines = text.split('\n')
    
    cleaned_lines = []
    previous_line = None
    
    for line in lines:
        line = line.strip()
        if line != previous_line and line != '':
            # Expand contractions
            line = contractions.fix(line)
            cleaned_lines.append(line)
            previous_line = line
    
    # Join lines back with a single newline
    return '\n'.join(cleaned_lines)

# Apply to your DataFrame
songs_df['final_lyrics_cleaned'] = songs_df['lyrics_normalized'].apply(clean_lyrics)

In [5]:
# Check first song
print(songs_df['final_lyrics_cleaned'].iloc[0])

he said he came from jamaica, he owned a couple acres a couple fake visas because he never got his papers gave up on love fucking with them heart breakers but he was getting money with the movers and the shakers he was mixed with a couple things, bald like a couple rings bricks in the condo and grams to sing-sing left arm, baby mother tatted 5-year bid up north when they ratted anyway i felt him, helped him, put him on lock, seat-belt him took him out to belgium, welcome bitches this pretty, that is seldom this box better than the box he was held in i am momma in that order, i call him daddy like daughters he like it when i get drunk, but i like it when he be sober that is top of the toppa, i never fuck with beginners i let him play with my pussy then lick it off of his fingers, i am in the zone they holler at me, but it is you, you, this is not high school me and my crew, we can slide through give it to you whenever you want, whip it whenever you want baby it is yours anywhere, everyw

In [6]:
# Load the FastText language identification model
model = fasttext.load_model("lid.176.bin")  # make sure this path points to where you saved lid.176.bin

# Function to check if the song is English
def is_english_song(text):
    # Replace \n with space to make it one line
    text_one_line = text.replace('\n', ' ')
    lang, prob = model.predict(text_one_line)
    return lang[0] == "__label__en" and prob[0] > 0.9


print(f"Number of English songs before filtering: {len(songs_df)}")
# Apply function and filter in-place
songs_df = songs_df[songs_df['final_lyrics_cleaned'].apply(is_english_song)].reset_index(drop=True)

print(f"Number of English songs after filtering: {len(songs_df)}")

Number of English songs before filtering: 520589
Number of English songs after filtering: 453384


In [7]:
# Check result
print(songs_df['final_lyrics_cleaned'].iloc[0])

he said he came from jamaica, he owned a couple acres a couple fake visas because he never got his papers gave up on love fucking with them heart breakers but he was getting money with the movers and the shakers he was mixed with a couple things, bald like a couple rings bricks in the condo and grams to sing-sing left arm, baby mother tatted 5-year bid up north when they ratted anyway i felt him, helped him, put him on lock, seat-belt him took him out to belgium, welcome bitches this pretty, that is seldom this box better than the box he was held in i am momma in that order, i call him daddy like daughters he like it when i get drunk, but i like it when he be sober that is top of the toppa, i never fuck with beginners i let him play with my pussy then lick it off of his fingers, i am in the zone they holler at me, but it is you, you, this is not high school me and my crew, we can slide through give it to you whenever you want, whip it whenever you want baby it is yours anywhere, everyw

In [8]:
stop_words = set(stopwords.words('english'))

lyrics_stopwords = {
    'yeah', 'oh', 'ooh', 'ah', 'na', 'la', 'ha',
    'gonna', 'wanna', 'gotta', 'chorus', 'verse',
    'hook', 'bridge', 'repeat', 'whoa', 'aye', 'ay', 
    'owhh', 'yeahh', 'ohoh', 'ahoh', 'ohohh', 'aha', 
    'ohh', 'uh', 'huh', 'mm', 'mmm', 'woo', 'yep', 'yup', 
    'hmm', 'hmmm', 'uhh', 'uhhh', 'huhh','ohoh', 'woooooa', 
    'da', 'ba', 'doo', 'blah', 'dum', 'ya', 'de', 'du', 'dee', 'bum',
    'ca', 'wo', 'wan', 'gon', 'ta', 'ai', 'uh', 'ya', 'yo', 'ai-ai',
    'la-la-la', 'na-na-na', 'la-la', 'na-na', 'oooooooooh', 'ooooooooooooooh',
    'mmmmmmmm', 'boo-boo-boo-boo-yah', 'oh-oh-oh-ohh'
}

all_stopwords = stop_words.union(lyrics_stopwords)

def clean_lyrics_keep_lines(text):
    lines = text.split('\n')  # keep each line
    cleaned_lines = []
    
    for line in lines:
        line = line.lower().strip()
        # remove punctuation/numbers, keep spaces
        line = re.sub(r'[^a-z\s]', ' ', line)
        # split into tokens, remove stopwords and single letters
        tokens = [t for t in line.split() if t and t not in all_stopwords and len(t) > 1]
        if tokens:  # only add non-empty lines
            cleaned_lines.append(' '.join(tokens))

    # final check to remove any leftover empty lines
    cleaned_lines = [line for line in cleaned_lines if line.strip()]
    return '\n'.join(cleaned_lines)

songs_df['final_lyrics_cleaned'] = songs_df['final_lyrics_cleaned'].apply(clean_lyrics_keep_lines)

#### Applying the model

In [10]:
documents = songs_df['final_lyrics_cleaned'].tolist()
print(f"Number of documents: {len(documents)}")
print(f"Sample document length: {len(documents[0])} characters")

Number of documents: 453384
Sample document length: 1230 characters


In [11]:
sample_df = songs_df.sample(20000, random_state=42).reset_index(drop=True)
sample_documents = sample_df['final_lyrics_cleaned'].tolist()

print(f"Number of sampled documents: {len(sample_documents)}")
print(f"Sample document length: {len(sample_documents[0])} characters")

Number of sampled documents: 20000
Sample document length: 750 characters


In [12]:
def chunk_lyrics(documents, min_words=12):
    """
    Splits a list of cleaned song strings into fixed-size word chunks.

    Parameters
    ----------
    documents : list of str   — one cleaned song per element
    min_words : int           — minimum words per chunk

    Returns
    -------
    chunks   : list of str   — all text chunks
    song_map : list of int   — index into `documents` for each chunk

    Works for both training (full list) and demo (single song):
        training : chunk_lyrics(sample_documents)
        demo     : chunk_lyrics([preprocess_single_song(raw_lyrics)])
    """
    chunks = []
    song_map = []
    for idx, song in enumerate(documents):
        lines = song.split('\n')
        buffer = []
        for line in lines:
            line = line.strip()
            if not line:
                continue
            buffer.extend(line.split())
            while len(buffer) >= min_words:
                chunks.append(' '.join(buffer[:min_words]))
                song_map.append(idx)
                buffer = buffer[min_words:]
        if buffer:
            if chunks and song_map[-1] == idx:
                chunks[-1] += ' ' + ' '.join(buffer)
            else:
                chunks.append(' '.join(buffer))
                song_map.append(idx)
    return chunks, song_map


sample_line_documents, song_map = chunk_lyrics(sample_documents, min_words=12)

print(f"Total chunks: {len(sample_line_documents)}")
print(f"Sample chunk: {sample_line_documents[0]}")

Total chunks: 183317
Sample chunk: little sleepy boy know time well hour bedtime long past though know


In [13]:
# Inspect chunks for the first song
first_song_index = 0
first_song_chunks = [
    chunk for chunk, idx in zip(sample_line_documents, song_map)
    if idx == first_song_index
]
for i, chunk in enumerate(first_song_chunks, start=1):
    print(f"Chunk {i}: {chunk}")

Chunk 1: little sleepy boy know time well hour bedtime long past though know
Chunk 2: fighting tell rub eyes fading fast fading fast run come see st
Chunk 3: judy comet roll across skies leave spray diamonds wake long see st
Chunk 4: judy comet sparkle eyes awake little boy lay body little boy close
Chunk 5: weary eyes nothing flashing fireflies well sang sang twice going sing three
Chunk 6: times going stay til resistance overcome cannot sing boy sleep well makes
Chunk 7: famous daddy look dumb run come see st judy comet roll across
Chunk 8: skies leave spray diamonds wake long see st judy comet sparkle eyes
Chunk 9: awake little boy little boy lay body little boy little boy close
Chunk 10: weary eyes nothing flashing fireflies little sleepy boy know time well hour bedtime long past though know fighting tell rub eyes fading fast


In [14]:
# 1. Initialise embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Compute embeddings in batches
batch_size = 256
all_embeddings = []
num_batches = math.ceil(len(sample_line_documents) / batch_size)

for i in range(num_batches):
    batch_lines = sample_line_documents[i*batch_size : (i+1)*batch_size]
    batch_embeddings = embedding_model.encode(batch_lines, show_progress_bar=True)
    all_embeddings.append(batch_embeddings)

all_embeddings = np.vstack([b.astype(np.float32) for b in all_embeddings])
print(f"Shape of embeddings: {all_embeddings.shape}")

# 3. Initialise BERTopic with UMAP
umap_model = UMAP(n_neighbors=60, n_components=8, low_memory=True)

topic_model = BERTopic(
    embedding_model=None,
    umap_model=umap_model,
    min_topic_size=100,
    top_n_words=10,
    nr_topics=None,
    calculate_probabilities=True,
    verbose=True
)

# 4. Fit BERTopic
topics, probs = topic_model.fit_transform(
    sample_line_documents,
    embeddings=all_embeddings
)

print("\nBefore outlier reduction:")
print(f"Number of topics: {len(set(topics)) - (1 if -1 in set(topics) else 0)}")

Batches: 100%|██████████| 1/1 [00:00<00:00,  9.12it/s]


Shape of embeddings: (183317, 384)


2026-04-14 15:12:18,637 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-14 15:15:30,809 - BERTopic - Dimensionality - Completed ✓
2026-04-14 15:15:30,826 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-14 15:18:16,360 - BERTopic - Cluster - Completed ✓
2026-04-14 15:18:16,406 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-14 15:18:18,405 - BERTopic - Representation - Completed ✓



Before outlier reduction:
Number of topics: 75


After training, we have:
- `topic_model` → the trained BERTopic model
- `topics` → topic ID for each chunk (before outlier reduction)
- `probs` → confidence scores
- `all_embeddings` → vectors (expensive to recompute — keep these!)
- `sample_line_documents` → actual text chunks
- `song_map` → mapping chunk index → song index in sample_df

In [15]:
topics_array = np.array(topics)
print(f"Outliers before reduction: {(topics_array == -1).sum()}")

# 5. Reduce outliers
topics_reduced = topic_model.reduce_outliers(
    sample_line_documents,
    topics,
    strategy="c-tf-idf"
)

print("\nAfter outlier reduction:")
print(f"Number of topics: {len(set(topics_reduced)) - (1 if -1 in set(topics_reduced) else 0)}")
print(f"Outliers remaining: {(np.array(topics_reduced) == -1).sum()}")

# 6. Update topic assignments in model
topic_model.update_topics(sample_line_documents, topics=topics_reduced)

# 7. Final distribution
print("\nUpdated topic distribution (top 10):")
print(pd.Series(topics_reduced).value_counts().sort_index().head(10))

Outliers before reduction: 143194


2026-04-14 15:18:22,889 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.



After outlier reduction:
Number of topics: 75
Outliers remaining: 13

Updated topic distribution (top 10):
-1       13
 0     9574
 1     8136
 2    15745
 3     3133
 4     3725
 5     6127
 6     5854
 7     3184
 8     4647
Name: count, dtype: int64


In [16]:
line_df_v6 = pd.DataFrame({
    "song_idx": song_map,
    "topic": topics_reduced
})

In [17]:
# View the top words per topic
topic_info = topic_model.get_topic_info()
print(topic_info.head(20))  # see top 20 topics

# Inspect a single topic
topic_id = 0
print(topic_model.get_topic(topic_id))

    Topic  Count                             Name  \
0      -1     13    -1_ravioli_pocketioli_yuk_ska   
1       0   9574        0_nigga_niggas_money_shit   
2       1   8136           1_god_lord_jesus_faith   
3       2  15745           2_love_heart_know_want   
4       3   3133          3_drink_wine_drunk_beer   
5       4   3725       4_mama_mother_father_daddy   
6       5   6127             5_baby_girl_want_got   
7       6   5854           6_ride_road_city_train   
8       7   3184  7_dream_dreams_dreaming_dreamed   
9       8   4647         8_believe_lies_truth_lie   
10      9   3168        9_fire_burn_burning_flame   
11     10  18098            10_think_time_go_back   
12     11   2577          11_sea_ocean_sail_waves   
13     12   1253   12_christmas_santa_bells_merry   
14     13   1805     13_dance_dancing_music_floor   
15     14   6102             14_eyes_look_see_eye   
16     15   2136     15_rain_clouds_storm_raining   
17     16   2023        16_door_doors_knock_fl

In [21]:
for i in range(0, 17):
    song = line_df_v6[line_df_v6['song_idx'] == i]
    if song.empty:
        print(f"Song {i}: No data\n")
        continue
    topic_counts = song['topic'].value_counts()
    dominant_topic = topic_counts.idxmax()
    print(f"Song {i}")
    print(topic_counts)
    print(f"Dominant topic: {dominant_topic}\n")

Song 0
topic
63    3
14    3
37    2
10    1
4     1
Name: count, dtype: int64
Dominant topic: 63

Song 1
topic
11    2
43    2
6     1
57    1
38    1
Name: count, dtype: int64
Dominant topic: 11

Song 2
topic
5     7
7     3
12    2
18    2
14    2
69    1
66    1
49    1
Name: count, dtype: int64
Dominant topic: 5

Song 3
topic
67    4
9     2
Name: count, dtype: int64
Dominant topic: 67

Song 4
topic
5    4
Name: count, dtype: int64
Dominant topic: 5

Song 5
topic
2     9
43    2
13    1
67    1
15    1
10    1
Name: count, dtype: int64
Dominant topic: 2

Song 6
topic
47    2
1     1
6     1
72    1
Name: count, dtype: int64
Dominant topic: 47

Song 7
topic
8     3
2     3
10    2
44    1
6     1
3     1
Name: count, dtype: int64
Dominant topic: 8

Song 8
topic
43    6
2     1
Name: count, dtype: int64
Dominant topic: 43

Song 9
topic
0     6
25    5
40    3
24    1
3     1
1     1
39    1
27    1
2     1
49    1
34    1
41    1
10    1
47    1
14    1
20    1
54    1
Name: count, 

In [22]:
# Flag songs where no topic appears more than once (potentially noisy)
song_chunks_grouped = line_df_v6.groupby('song_idx')
noisy_songs = song_chunks_grouped.filter(lambda x: x['topic'].value_counts().max() == 1)
print(f"Songs with no dominant topic: {noisy_songs['song_idx'].nunique()}")

Songs with no dominant topic: 2719


In [23]:
for i in range(len(topic_model.get_topics())):
    print(topic_model.get_topic(i))

[('nigga', 0.05331703844190332), ('niggas', 0.04214219221140003), ('money', 0.04024288630273298), ('shit', 0.03602348360582495), ('fuck', 0.03118246376013666), ('bitch', 0.031112635569588716), ('get', 0.016841975323415467), ('got', 0.015597848967848384), ('ass', 0.015559407706625249), ('like', 0.01339042987722244)]
[('god', 0.049206701820910026), ('lord', 0.03291433272766896), ('jesus', 0.030233078221298715), ('faith', 0.01894925954972982), ('sing', 0.016778461855408634), ('pray', 0.015995905872889807), ('praise', 0.015238901961963942), ('king', 0.015228216404187946), ('holy', 0.01521749553317872), ('name', 0.014820678768888965)]
[('love', 0.04807395800946726), ('heart', 0.02678264499365551), ('know', 0.018498348678230858), ('want', 0.018196003365718923), ('need', 0.01702908598948096), ('give', 0.015270128346115035), ('say', 0.014612408402024827), ('feel', 0.013100027131635699), ('never', 0.012423258566677574), ('really', 0.011217469715108024)]
[('drink', 0.05647241939130575), ('wine',

In [24]:
print(f"Topics distribution:\n{pd.Series(np.array(topics_reduced)).value_counts().sort_values(ascending=False).head(100)}")

Topics distribution:
 10    18098
 2     15745
 0      9574
 1      8136
 5      6127
       ...  
 68      531
 50      353
 70      304
 66      294
-1        13
Name: count, Length: 76, dtype: int64


## Evaluation Metrics

In [25]:
# ── Coherence Score ──────────────────────────────────────────────────────────
tokenized_docs = [doc.split() for doc in sample_line_documents]
dictionary = Dictionary(tokenized_docs)

topics_dict = topic_model.get_topics()
topic_words = []
topic_ids_eval = []

for topic_id, words_weights in topics_dict.items():
    if topic_id != -1 and len(words_weights) >= 10:
        words = [word for word, _ in words_weights[:15]]
        topic_words.append(words)
        topic_ids_eval.append(topic_id)

coherence_model_cv = CoherenceModel(
    topics=topic_words,
    texts=tokenized_docs,
    dictionary=dictionary,
    coherence='c_v'
)
overall_coherence = coherence_model_cv.get_coherence()
print(f"Overall Coherence Score: {overall_coherence:.4f}")


# ── Topic Diversity ───────────────────────────────────────────────────────────
def compute_topic_diversity(topic_model, top_k=15):
    topics = topic_model.get_topics()
    all_words = []
    for topic_id, words in topics.items():
        if topic_id == -1:
            continue
        all_words.extend([word for word, _ in words[:top_k]])
    return len(set(all_words)) / len(all_words)

diversity_score = compute_topic_diversity(topic_model)
print(f"Topic Diversity Score: {diversity_score:.4f}")

Overall Coherence Score: 0.4554
Topic Diversity Score: 0.8053
